# FIT · Step 1 & 2: Pull and clean one season

**FIT** is a basketball impact metric I'm building. It measures how much a player adds and how well players fit together.

This notebook is the data pipeline:

1. **Pull** every game of a season from NBA.com: the schedule, the play-by-play, and the rotations (who checked in and out, and when).
2. **Clean** it into **stints**: stretches of a game where the same 10 players are on the floor. For each stint we keep who was out there, how many points each team scored, and how many possessions it lasted.
3. **Check** every game against the official box score. Points have to add up to the final score, every stint needs exactly 5 players per team, and the stints have to cover the whole game. Games that fail get flagged and left out.

Output: `data/stints.csv`, the input for RAPM and later for FIT.

In [ ]:
%pip install nba_api
%pip install pandas, numpy

In [10]:
%pip install nba_api
%pip install pandas
%pip install matplotlib
%pip install seaborn
import os
import re
import time

import pandas as pd
import numpy as np
from nba_api.stats import endpoints

from matplotlib import pyplot as plt
import seaborn as sns; sns.set_style('whitegrid')

pd.set_option('display.max_columns', 100)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
   ------------------- -------------------- 143.4/294.9 kB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 294.9/294.9 kB 4.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
SEASON = '2025-26'

N_GAMES = 20          # start small to make sure everything works, then set to None for the full season
SLEEP = 1.0           # pause between requests so NBA.com doesn't block us

DATA_DIR = 'data'
RAW_PBP_DIR = os.path.join(DATA_DIR, 'raw', 'pbp')
RAW_ROT_DIR = os.path.join(DATA_DIR, 'raw', 'rotation')

os.makedirs(RAW_PBP_DIR, exist_ok = True)
os.makedirs(RAW_ROT_DIR, exist_ok = True)

## Step 1: Pull the data

First the schedule. `LeagueGameFinder` gives one row per team per game, so every game shows up twice. `vs.` in the matchup means home, `@` means away.

In [14]:
games = endpoints.leaguegamefinder.LeagueGameFinder(season_nullable = SEASON,
                                                     league_id_nullable = '00',
                                                     season_type_nullable = 'Regular Season').league_game_finder_results.get_data_frame()

games['IS_HOME'] = games['MATCHUP'].str.contains('vs.', regex = False)

home = games.loc[games['IS_HOME'], ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'TEAM_ABBREVIATION', 'PTS']]
away = games.loc[~games['IS_HOME'], ['GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'PTS']]

home.columns = ['GAME_ID', 'GAME_DATE', 'HOME_ID', 'HOME', 'HOME_PTS']
away.columns = ['GAME_ID', 'AWAY_ID', 'AWAY', 'AWAY_PTS']

game_list = home.merge(away, on = 'GAME_ID', how = 'inner').sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop = True)

game_list.to_csv(os.path.join(DATA_DIR, 'games.csv'), index = False)

print(f'Games found: {len(game_list)}')
game_list.head()

Games found: 1225


,GAME_ID,GAME_DATE,HOME_ID,HOME,HOME_PTS,AWAY_ID,AWAY,AWAY_PTS
0,0022500001,2025-10-21,1610612760,OKC,125,1610612745,HOU,124
1,0022500002,2025-10-21,1610612747,LAL,109,1610612744,GSW,119
2,0022500003,2025-10-22,1610612752,NYK,119,1610612739,CLE,111
3,0022500004,2025-10-22,1610612742,DAL,92,1610612759,SAS,125
4,0022500080,2025-10-22,1610612766,CHA,136,1610612751,BKN,117


Now the play-by-play and the rotation for every game. Each game is saved to its own file, so if the download stops (NBA.com times out sometimes) you can rerun the cell and it picks up where it left off.

In [15]:
def call_api(func, game_id, tries = 3):

    for attempt in range(tries):

        try:
            return func(game_id)

        except Exception as e:
            print(f'{game_id}: attempt {attempt + 1} failed ({e})')
            time.sleep(5 * (attempt + 1))

    return None


def get_pbp(game_id):
    return endpoints.playbyplayv3.PlayByPlayV3(game_id = game_id, timeout = 60).play_by_play.get_data_frame()


def get_rotation(game_id):

    rotation = endpoints.gamerotation.GameRotation(game_id = game_id, timeout = 60)

    home_rot = rotation.home_team.get_data_frame()
    away_rot = rotation.away_team.get_data_frame()

    home_rot['SIDE'] = 'HOME'
    away_rot['SIDE'] = 'AWAY'

    return pd.concat([home_rot, away_rot])

In [ ]:
game_ids = game_list['GAME_ID'].tolist()

if N_GAMES is not None:
    game_ids = game_ids[:N_GAMES]

failed = []
start_time = time.time()

for i, game_id in enumerate(game_ids):

    pbp_path = os.path.join(RAW_PBP_DIR, f'{game_id}.csv')
    rot_path = os.path.join(RAW_ROT_DIR, f'{game_id}.csv')

    if os.path.exists(pbp_path) and os.path.exists(rot_path):
        continue

    pbp = call_api(get_pbp, game_id)
    time.sleep(SLEEP)
    rot = call_api(get_rotation, game_id)
    time.sleep(SLEEP)

    if pbp is None or rot is None or len(pbp) == 0 or len(rot) == 0:
        failed.append(game_id)
        continue

    pbp.to_csv(pbp_path, index = False)
    rot.to_csv(rot_path, index = False)

    if (i + 1) % 50 == 0:
        print(f'{i + 1} / {len(game_ids)} games downloaded ({(time.time() - start_time) / 60:.1f} min)')

print(f'Done. Games that failed to download: {len(failed)}')

## Step 2: Clean the data into stints

Time is measured in **tenths of a second since tip-off**, the same unit the rotation data uses. One quarter is 7,200 and a full game is 28,800. Overtime periods are 3,000 each.

The play-by-play clock counts down (`PT11M48.00S` means 11:48 left in the period), so it has to be converted into time since tip-off.

In [ ]:
def clock_to_elapsed(period, clock):

    found = re.findall(r'PT(\d+)M([\d.]+)S', clock) if isinstance(clock, str) else []

    if len(found) == 0:
        return np.nan

    minutes, seconds = found[0]
    remaining = (int(minutes) * 60 + float(seconds)) * 10

    if period <= 4:
        start, length = (period - 1) * 7200, 7200
    else:
        start, length = 28800 + (period - 5) * 3000, 3000

    return start + length - remaining


print(clock_to_elapsed(1, 'PT12M00.00S'))   # tip-off -> 0
print(clock_to_elapsed(2, 'PT06M00.00S'))   # halfway through the 2nd -> 10800
print(clock_to_elapsed(5, 'PT00M00.00S'))   # end of the first overtime -> 31800

For every play we need the points scored and the pieces of the possession estimate:

**Possessions = FGA + 0.44 × FTA − offensive rebounds + turnovers**

An offensive rebound is a rebound by the same team that just missed.

In [ ]:
def prep_pbp(pbp, home_id):

    pbp = pbp.sort_values('actionNumber').copy()

    pbp['ELAPSED'] = pbp.apply(lambda x: clock_to_elapsed(x.period, x.clock), axis = 1)
    pbp['ELAPSED'] = pbp['ELAPSED'].ffill().fillna(0)

    # the score is only filled in on scoring plays, so carry it forward
    pbp['scoreHome'] = pd.to_numeric(pbp['scoreHome'], errors = 'coerce').ffill().fillna(0)
    pbp['scoreAway'] = pd.to_numeric(pbp['scoreAway'], errors = 'coerce').ffill().fillna(0)

    pbp['PTS_HOME'] = pbp['scoreHome'].diff().fillna(pbp['scoreHome'])
    pbp['PTS_AWAY'] = pbp['scoreAway'].diff().fillna(pbp['scoreAway'])

    pbp['IS_HOME'] = pbp['teamId'] == home_id

    pbp['FGA'] = pbp['isFieldGoal'] == 1
    pbp['FTA'] = pbp['actionType'] == 'Free Throw'
    pbp['TOV'] = pbp['actionType'] == 'Turnover'

    pbp['MISS'] = (pbp['actionType'] == 'Missed Shot') | (pbp['FTA'] & pbp['description'].str.contains('MISS', na = False))

    last_miss_team = pbp['teamId'].where(pbp['MISS']).ffill()
    pbp['OREB'] = (pbp['actionType'] == 'Rebound') & (pbp['teamId'] == last_miss_team)

    return pbp


def count_possessions(rows):

    counts = rows.groupby('STINT')[['FGA', 'FTA', 'OREB', 'TOV']].sum()

    return counts['FGA'] + 0.44 * counts['FTA'] - counts['OREB'] + counts['TOV']

Building the stints:

- Every time someone checks in or out, a new stint starts.
- The players on the floor are the ones checked in at the middle of the stint.
- A play that happens at the exact moment of a substitution (like free throws right after a sub) is credited to the lineup that was on the floor **before** the sub.

In [ ]:
def build_stints(game_id, home_id, away_id):

    pbp = pd.read_csv(os.path.join(RAW_PBP_DIR, f'{game_id}.csv'))
    rot = pd.read_csv(os.path.join(RAW_ROT_DIR, f'{game_id}.csv'))

    pbp = prep_pbp(pbp, home_id)

    boundaries = np.unique(np.concatenate([rot['IN_TIME_REAL'], rot['OUT_TIME_REAL']]))

    stints = []

    for start, end in zip(boundaries[:-1], boundaries[1:]):

        mid = (start + end) / 2

        on_floor = rot.loc[(rot['IN_TIME_REAL'] <= mid) & (rot['OUT_TIME_REAL'] > mid)]

        stints.append({'GAME_ID': game_id,
                       'START': start,
                       'END': end,
                       'HOME_PLAYERS': sorted(on_floor.loc[on_floor['SIDE'] == 'HOME', 'PERSON_ID'].astype(int)),
                       'AWAY_PLAYERS': sorted(on_floor.loc[on_floor['SIDE'] == 'AWAY', 'PERSON_ID'].astype(int))})

    stints = pd.DataFrame(stints)

    pbp['STINT'] = (np.searchsorted(boundaries, pbp['ELAPSED'], side = 'left') - 1).clip(0, len(stints) - 1)

    points = pbp.groupby('STINT')[['PTS_HOME', 'PTS_AWAY']].sum()
    stints = stints.join(points).fillna({'PTS_HOME': 0, 'PTS_AWAY': 0})

    stints['POSS_HOME'] = count_possessions(pbp.loc[pbp['IS_HOME']])
    stints['POSS_AWAY'] = count_possessions(pbp.loc[pbp['teamId'] == away_id])
    stints[['POSS_HOME', 'POSS_AWAY']] = stints[['POSS_HOME', 'POSS_AWAY']].fillna(0).clip(lower = 0)

    stints['HOME_ID'] = home_id
    stints['AWAY_ID'] = away_id
    stints['SECONDS'] = (stints['END'] - stints['START']) / 10
    stints['PERIODS'] = pbp['period'].max()

    # score margin when the stint starts, used to flag garbage time
    margin = (stints['PTS_HOME'] - stints['PTS_AWAY']).cumsum()
    stints['MARGIN_START'] = margin.shift().fillna(0)
    stints['GARBAGE'] = (stints['START'] >= 25800) & (stints['START'] < 28800) & (stints['MARGIN_START'].abs() >= 20)

    stints['N_HOME'] = stints['HOME_PLAYERS'].apply(len)
    stints['N_AWAY'] = stints['AWAY_PLAYERS'].apply(len)

    for side, col in [('H', 'HOME_PLAYERS'), ('A', 'AWAY_PLAYERS')]:
        for k in range(5):
            stints[f'{side}{k + 1}'] = stints[col].apply(lambda p: p[k] if len(p) == 5 else np.nan)

    players = rot[['PERSON_ID', 'PLAYER_FIRST', 'PLAYER_LAST', 'TEAM_ID']].drop_duplicates()

    return stints.drop(columns = ['HOME_PLAYERS', 'AWAY_PLAYERS']), players

The checks. This is the most important part of the cleaning: if the numbers we rebuilt don't match the official ones, the game doesn't go into the model.

In [ ]:
def check_game(stints, game):

    expected_end = 28800 + max(stints['PERIODS'].iloc[0] - 4, 0) * 3000
    real_stints = stints.loc[stints['SECONDS'] > 0]

    return {'GAME_ID': game['GAME_ID'],
            'POINTS_OK': (stints['PTS_HOME'].sum() == game['HOME_PTS']) and (stints['PTS_AWAY'].sum() == game['AWAY_PTS']),
            'LINEUPS_OK': ((real_stints['N_HOME'] == 5) & (real_stints['N_AWAY'] == 5)).all(),
            'TIME_OK': (stints['START'].min() == 0) and (stints['END'].max() == expected_end)}

In [ ]:
all_stints = []
all_players = []
checks = []

for _, game in game_list.loc[game_list['GAME_ID'].isin(game_ids)].iterrows():

    game_id = game['GAME_ID']

    if not os.path.exists(os.path.join(RAW_PBP_DIR, f'{game_id}.csv')):
        continue

    stints, players = build_stints(game_id, game['HOME_ID'], game['AWAY_ID'])

    all_stints.append(stints)
    all_players.append(players)
    checks.append(check_game(stints, game))

stints = pd.concat(all_stints).reset_index(drop = True)
checks = pd.DataFrame(checks)

checks['ALL_OK'] = checks['POINTS_OK'] & checks['LINEUPS_OK'] & checks['TIME_OK']

print(f'Games checked: {len(checks)}')
print(f'Points match the final score: {checks["POINTS_OK"].sum()}')
print(f'5 players per team in every stint: {checks["LINEUPS_OK"].sum()}')
print(f'Stints cover the whole game: {checks["TIME_OK"].sum()}')
print(f'Pass all checks: {checks["ALL_OK"].sum()}')

In [ ]:
bad_games = checks.loc[~checks['ALL_OK']]

bad_games.to_csv(os.path.join(DATA_DIR, 'bad_games.csv'), index = False)

bad_games

Keep only the games that passed every check, and save the stints and the player names.

In [ ]:
good_ids = checks.loc[checks['ALL_OK'], 'GAME_ID']

clean = stints.loc[stints['GAME_ID'].isin(good_ids) & (stints['SECONDS'] > 0)].copy()

player_cols = ['H1', 'H2', 'H3', 'H4', 'H5', 'A1', 'A2', 'A3', 'A4', 'A5']
clean[player_cols] = clean[player_cols].astype(int)

clean.to_csv(os.path.join(DATA_DIR, 'stints.csv'), index = False)

players = pd.concat(all_players).drop_duplicates('PERSON_ID', keep = 'last')
players['PLAYER'] = players['PLAYER_FIRST'] + ' ' + players['PLAYER_LAST']
players[['PERSON_ID', 'PLAYER', 'TEAM_ID']].to_csv(os.path.join(DATA_DIR, 'players.csv'), index = False)

print(f'Clean stints: {len(clean)} from {clean["GAME_ID"].nunique()} games')
print(f'Possessions: {clean["POSS_HOME"].sum() + clean["POSS_AWAY"].sum():,.0f}')
clean.head()

## Sanity check

Same idea as checking LeBron in the MVP project: look at a player we know and see if the numbers make sense. Minutes per game should look close to what Basketball-Reference shows.

In [ ]:
long = clean.melt(id_vars = ['GAME_ID', 'SECONDS'], value_vars = player_cols, value_name = 'PERSON_ID')

minutes = long.groupby('PERSON_ID').agg(GAMES = ('GAME_ID', 'nunique'), MIN = ('SECONDS', 'sum')).reset_index()
minutes['MIN'] = minutes['MIN'] / 60
minutes['MPG'] = minutes['MIN'] / minutes['GAMES']

minutes = minutes.merge(players[['PERSON_ID', 'PLAYER']], on = 'PERSON_ID', how = 'left')

minutes.sort_values('MIN', ascending = False).head(10)

In [ ]:
minutes.loc[minutes['PLAYER'] == 'Victor Wembanyama']

In [ ]:
plt.figure(figsize = (8, 5))

sns.histplot(clean['SECONDS'] / 60, bins = 40)

plt.xlabel('Stint length (minutes)')
plt.ylabel('Number of stints')
plt.title(f'How long the same 10 players stay on the floor ({SEASON})')
plt.show()